In [ ]:
import gzip
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_IMAGES_PATH = "train-images-idx3-ubyte.gz"
TRAIN_LABELS_PATH = "train-labels-idx1-ubyte.gz"

TEST_IMAGES_PATH = "t10k-images-idx3-ubyte.gz"
TEST_LABELS_PATH = "t10k-labels-idx1-ubyte.gz"


def read_images_from_gzip(filename):
    with gzip.open(filename, "rb") as f:
        magic_number = int.from_bytes(f.read(4), byteorder="big")
        number_of_images = int.from_bytes(f.read(4), byteorder="big")
        rows = int.from_bytes(f.read(4), byteorder="big")
        cols = int.from_bytes(f.read(4), byteorder="big")

        print("Файл изображений:", filename)
        print("Magic number:", magic_number)
        print("Количество изображений:", number_of_images)
        print("Размер изображения:", rows, "x", cols)
        print()

        image_data = f.read()
        images = np.frombuffer(image_data, dtype=np.uint8)
        images = images.reshape(number_of_images, rows, cols)

        return images


def read_labels_from_gzip(filename):
    with gzip.open(filename, "rb") as f:
        magic_number = int.from_bytes(f.read(4), byteorder="big")
        number_of_labels = int.from_bytes(f.read(4), byteorder="big")

        print("Файл меток:", filename)
        print("Magic number:", magic_number)
        print("Количество меток:", number_of_labels)
        print()

        label_data = f.read()
        labels = np.frombuffer(label_data, dtype=np.uint8)

        return labels


class MNISTDataset(Dataset):

    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

        self.images = self.images.unsqueeze(1)
        self.images = self.images / 255.0
        self.images = (self.images - 0.1307) / 0.3081

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        return image, label


def get_data_loaders():
    train_images = read_images_from_gzip(TRAIN_IMAGES_PATH)
    train_labels = read_labels_from_gzip(TRAIN_LABELS_PATH)

    test_images = read_images_from_gzip(TEST_IMAGES_PATH)
    test_labels = read_labels_from_gzip(TEST_LABELS_PATH)

    train_dataset = MNISTDataset(train_images, train_labels)
    test_dataset = MNISTDataset(test_images, test_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    return train_loader, test_loader



class ManualConv2d(nn.Module):

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        self.weight = nn.Parameter(
            torch.randn(out_channels, in_channels, kernel_size, kernel_size)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_channels)
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        batch_size, channels, height, width = x.shape

        patches = F.unfold(
            x,
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=self.padding
        )

        weight_flat = self.weight.view(self.out_channels, -1)

        output = torch.matmul(weight_flat, patches)

        output = output + self.bias.view(1, -1, 1)

        output_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        output_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1

        output = output.view(
            batch_size,
            self.out_channels,
            output_height,
            output_width
        )

        return output


class ManualAvgPool2d(nn.Module):

    def __init__(self, kernel_size, stride):
        super().__init__()

        self.kernel_size = kernel_size
        self.stride = stride

    def forward(self, x):
        batch_size, channels, height, width = x.shape

        patches = F.unfold(
            x,
            kernel_size=self.kernel_size,
            stride=self.stride
        )

        patches = patches.view(
            batch_size,
            channels,
            self.kernel_size * self.kernel_size,
            -1
        )

        output = patches.mean(dim=2)

        output_height = (height - self.kernel_size) // self.stride + 1
        output_width = (width - self.kernel_size) // self.stride + 1

        output = output.view(
            batch_size,
            channels,
            output_height,
            output_width
        )

        return output


class ManualLinear(nn.Module):

    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(
            torch.randn(out_features, in_features)
        )

        self.bias = nn.Parameter(
            torch.zeros(out_features)
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        output = x @ self.weight.t() + self.bias

        return output


class ManualLeNet5(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = ManualConv2d(
            in_channels=1,
            out_channels=6,
            kernel_size=5
        )

        self.pool1 = ManualAvgPool2d(
            kernel_size=2,
            stride=2
        )

        self.conv2 = ManualConv2d(
            in_channels=6,
            out_channels=16,
            kernel_size=5
        )

        self.pool2 = ManualAvgPool2d(
            kernel_size=2,
            stride=2
        )

        self.fc1 = ManualLinear(16 * 5 * 5, 120)
        self.fc2 = ManualLinear(120, 84)
        self.fc3 = ManualLinear(84, 10)

    def forward(self, x):
        x = F.pad(x, pad=(2, 2, 2, 2))

        x = self.conv1(x)
        x = torch.tanh(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = torch.tanh(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = torch.tanh(x)

        x = self.fc2(x)
        x = torch.tanh(x)

        x = self.fc3(x)

        return x


def train_model(model, train_loader, criterion, optimizer):
    model.train()

    total_loss = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    return average_loss


def evaluate_model(model, test_loader, criterion):
    model.eval()

    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = torch.argmax(outputs, dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    average_loss = total_loss / len(test_loader)
    accuracy = accuracy_score(all_labels, all_predictions)

    return average_loss, accuracy, all_labels, all_predictions


def run_experiment(model_class, model_name, train_loader, test_loader):
    print()
    print("=" * 60)
    print("Запуск модели:", model_name)
    print("=" * 60)

    torch.manual_seed(42)

    model = model_class().to(DEVICE)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    train_losses = []
    test_losses = []
    test_accuracies = []

    final_labels = []
    final_predictions = []

    start_time = time.time()

    for epoch in range(EPOCHS):
        train_loss = train_model(
            model,
            train_loader,
            criterion,
            optimizer
        )

        test_loss, test_accuracy, all_labels, all_predictions = evaluate_model(
            model,
            test_loader,
            criterion
        )

        train_losses.append(train_loss)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        final_labels = all_labels
        final_predictions = all_predictions

        print(
            f"{model_name} | "
            f"Epoch [{epoch + 1}/{EPOCHS}] "
            f"Train Loss: {train_loss:.4f} "
            f"Test Loss: {test_loss:.4f} "
            f"Accuracy: {test_accuracy:.4f}"
        )

    end_time = time.time()

    result = {
        "model_name": model_name,
        "train_losses": train_losses,
        "test_losses": test_losses,
        "test_accuracies": test_accuracies,
        "final_train_loss": train_losses[-1],
        "final_test_loss": test_losses[-1],
        "final_accuracy": test_accuracies[-1],
        "training_time": end_time - start_time,
        "all_labels": final_labels,
        "all_predictions": final_predictions
    }

    return result


def plot_comparison(library_result, manual_result):
    epochs_range = range(1, EPOCHS + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs_range,
        library_result["train_losses"],
        marker="o",
        label="Library Train Loss"
    )
    plt.plot(
        epochs_range,
        manual_result["train_losses"],
        marker="o",
        label="Manual Train Loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Сравнение Train Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs_range,
        library_result["test_losses"],
        marker="o",
        label="Library Test Loss"
    )
    plt.plot(
        epochs_range,
        manual_result["test_losses"],
        marker="o",
        label="Manual Test Loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Сравнение Test Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs_range,
        library_result["test_accuracies"],
        marker="o",
        label="Library Accuracy"
    )
    plt.plot(
        epochs_range,
        manual_result["test_accuracies"],
        marker="o",
        label="Manual Accuracy"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Сравнение Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_confusion_matrices(library_result, manual_result):
    library_matrix = confusion_matrix(
        library_result["all_labels"],
        library_result["all_predictions"]
    )

    manual_matrix = confusion_matrix(
        manual_result["all_labels"],
        manual_result["all_predictions"]
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    display = ConfusionMatrixDisplay(
        confusion_matrix=library_matrix,
        display_labels=list(range(10))
    )
    display.plot(ax=ax, values_format="d")
    plt.title("Матрица ошибок: библиотечная LeNet-5")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 6))
    display = ConfusionMatrixDisplay(
        confusion_matrix=manual_matrix,
        display_labels=list(range(10))
    )
    display.plot(ax=ax, values_format="d")
    plt.title("Матрица ошибок: ручная LeNet-5")
    plt.show()


def print_comparison_table(library_result, manual_result):
    print()
    print("=" * 90)
    print("Итоговое сравнение")
    print("=" * 90)

    print(
        f"{'Модель':<30}"
        f"{'Train Loss':<15}"
        f"{'Test Loss':<15}"
        f"{'Accuracy':<15}"
        f"{'Время, сек':<15}"
    )

    print("-" * 90)

    print(
        f"{library_result['model_name']:<30}"
        f"{library_result['final_train_loss']:<15.4f}"
        f"{library_result['final_test_loss']:<15.4f}"
        f"{library_result['final_accuracy']:<15.4f}"
        f"{library_result['training_time']:<15.2f}"
    )

    print(
        f"{manual_result['model_name']:<30}"
        f"{manual_result['final_train_loss']:<15.4f}"
        f"{manual_result['final_test_loss']:<15.4f}"
        f"{manual_result['final_accuracy']:<15.4f}"
        f"{manual_result['training_time']:<15.2f}"
    )


def main():
    print("Используемое устройство:", DEVICE)
    print()

    train_loader, test_loader = get_data_loaders()

    library_result = run_experiment(
        LibraryLeNet5,
        "Библиотечная LeNet-5",
        train_loader,
        test_loader
    )

    manual_result = run_experiment(
        ManualLeNet5,
        "Ручная LeNet-5",
        train_loader,
        test_loader
    )

    print_comparison_table(
        library_result,
        manual_result
    )

    plot_comparison(
        library_result,
        manual_result
    )

    plot_confusion_matrices(
        library_result,
        manual_result
    )


if __name__ == "__main__":
    main()